[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Surjasa/teen-mental-health-analysis/blob/main/notebooks/01_data_inspection_preprocessing.ipynb)

# 01 Data Inspection and Preprocessing

This notebook keeps the modeling workflow separate from inference. It loads the raw CSV, inspects the columns, selects `depression_label` as the target, splits the data, encodes categorical features, scales numeric features, applies SMOTE only to the training data, and saves the balanced training set. Also cleans the dataset by performing data cleaning.

In [ ]:
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.over_sampling import SMOTE

In [ ]:
from google.colab import files
print('Select dataset (CSV) to upload when prompted.')
uploaded = files.upload()
if uploaded:
    fname = list(uploaded.keys())[0]
    df = pd.read_csv(fname)
print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
display(df.head())
print(df.info())

In [ ]:
target_col = 'depression_label'
X = df.drop(target_col, axis=1)
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train counts:')
print(y_train.value_counts())
print('y_test counts:')
print(y_test.value_counts())

In [ ]:
# Separate feature columns by type so we can preprocess them correctly.
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=['object']).columns.tolist()
print('Numeric columns:', numeric_cols)
print('Categorical columns:', categorical_cols)

# Standardize numeric columns and one-hot encode categorical columns.
# This converts everything into numeric form before SMOTE runs.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ]
)
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)
print('Encoded train shape:', X_train_prepared.shape)
print('Encoded test shape :', X_test_prepared.shape)

# SMOTE balances only the training target labels.
# It does not touch the test set.
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_prepared, y_train)
print('Before SMOTE:')
print(y_train.value_counts())
print('After SMOTE:')
print(y_train_res.value_counts())
print('Prepared train shape:', X_train_prepared.shape)
print('Balanced train shape:', X_train_res.shape)

In [ ]:
from google.colab import files

feature_names = preprocessor.get_feature_names_out()
train_bal = pd.DataFrame(X_train_res, columns=feature_names)
train_bal[target_col] = y_train_res
output_file = 'train_balanced.csv'
train_bal.to_csv(output_file, index=False)
files.download(output_file)
print(f'Created and downloaded {output_file}')

In [ ]:
# Data cleaning helper
def clean_dataset(df_in):
    """Basic cleaning: drop duplicates, report and fill missing values.
    Numeric columns: fill with median. Categorical: fill with mode.
    Returns a cleaned copy of the dataframe."""
    df_clean = df_in.copy()
    # drop exact duplicates
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    after = len(df_clean)
    print(f'Dropped {before-after} duplicate rows')

    print('\nMissing values per column:')
    print(df_clean.isna().sum())

    num_cols = df_clean.select_dtypes(include=['number']).columns
    cat_cols = df_clean.select_dtypes(include=['object', 'category']).columns

    for c in num_cols:
        if df_clean[c].isna().any():
            med = df_clean[c].median()
            df_clean[c] = df_clean[c].fillna(med)
            print(f'Filled NA in numeric {c} with median={med}')

    for c in cat_cols:
        if df_clean[c].isna().any():
            mode = df_clean[c].mode()
            fill = mode.iloc[0] if not mode.empty else ''
            df_clean[c] = df_clean[c].fillna(fill)
            print(f"Filled NA in categorical {c} with mode='{fill}'")

    return df_clean

# Example usage:
# df = clean_dataset(df)
